# Burn cost workflow demo: optional manual adjustment

Use this notebook only for business uplifts or reductions after fitting.

Browse the published history, open the current deployment by default,
and express business uplifts or reductions as replayable relative rules.
Publication creates an immutable `MANUAL_EDIT` child. Deployment remains
an explicit final decision.


In [ ]:
DATABASE_MODE = "remote"  # "local" or "remote"
RUNTIME_MODULE = "demo_sql_runtime"  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = "PricingNotebookDemo"
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "DEMO_BURN_COST"  # Set to None to select by label only.
MODEL_LABEL = "Burn cost workflow demo"
DEPLOYMENT_SLOT = "DEMO_BURN_COST_ONLY"
SOURCE_SELECTOR = "deployed"  # "deployed" or "latest"
PACKAGE_VERSION = None  # Choose after reading the package list.
POLICY_SOURCE_PACKAGE_VERSION = None  # Reuse a prior MANUAL_EDIT policy when set.

POLICY_NAME = "DEMO_BURN_COST manual adjustment"
POLICY_VERSION = 1
CARRY_FORWARD = True
POLICY_REASON = ""

# Each row selects either levels or an x_range and multiplies their relativity.
# Python performs the log-link conversion. Add as many non-overlapping rows as needed.
MANUAL_ADJUSTMENTS = [
    # {
    #     "feature": "feature_name",
    #     "levels": ["level_a", "level_b"],
    #     "factor": 1.05,
    #     "reason": "Approved business rationale",
    # },
    # {
    #     "feature": "continuous_feature",
    #     "x_range": [10, 20],
    #     "factor": 0.97,
    #     "reason": "Approved business rationale",
    # },
]

DEPLOY_AFTER_PUBLISH = False
DEPLOYMENT_REASON = ""  # Explain the decision after reviewing.

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "pricing_models").is_dir()
)

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pricing_pipeline.notebook import (
    ManualAdjustmentPolicy,
    apply_manual_adjustment_policy,
    connect,
    deploy_model_version,
    list_model_versions,
    load_registered_model,
    manual_adjustment_policy_from_candidate,
    load_model_version,
    open_deployed_candidate,
    publish_manual_adjustment,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/burn_cost_demo"

## Connect and browse every published version


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)

In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_model_versions(pricing, model=model, technical=True)
if versions.empty:
    raise LookupError("No published package versions were found.")
inventory = versions.loc[
    :,
    [
        "rate_package_id",
        "package_version",
        "recipe_revision",
        "model_kind",
        "data_as_of_date",
        "completed_ts",
        "parent_package_version",
        "current_rate_package_id",
    ],
].copy()
inventory.insert(
    0,
    "deployed",
    inventory["rate_package_id"].eq(inventory["current_rate_package_id"]),
)
inventory.insert(1, "model_label", MODEL_LABEL)
display(
    inventory.loc[
        :,
        [
            "package_version",
            "recipe_revision",
            "deployed",
            "model_kind",
            "data_as_of_date",
            "completed_ts",
            "parent_package_version",
        ],
    ].rename(
        columns={
            "package_version": "Package",
            "recipe_revision": "Model version",
            "deployed": "Current champion?",
            "model_kind": "Kind",
            "data_as_of_date": "Data as of",
            "completed_ts": "Fitted at",
            "parent_package_version": "Parent package",
        }
    )
)

## Select and verify the exact source package


In [ ]:
if PACKAGE_VERSION is not None:
    selected_package_version = int(PACKAGE_VERSION)
    if selected_package_version not in set(versions["package_version"].astype(int)):
        raise ValueError("PACKAGE_VERSION is not in the displayed history.")
    reviewed = load_model_version(
        pricing,
        model=model,
        package_version=selected_package_version,
    )
elif SOURCE_SELECTOR == "deployed":
    reviewed = open_deployed_candidate(pricing, model=model)
elif SOURCE_SELECTOR == "latest":
    reviewed = load_model_version(
        pricing,
        model=model,
        package_version=int(versions.iloc[0]["package_version"]),
    )
else:
    raise ValueError("SOURCE_SELECTOR must be 'deployed' or 'latest'.")
display(
    {
        "Package": reviewed.package_version,
        "Model version": reviewed.recipe_revision,
        "Kind": reviewed.technical.get("model_kind"),
        "Data as of": reviewed.technical.get("data_as_of_date"),
        "Parent package": reviewed.technical.get("parent_package_version"),
    }
)

## Declare the replayable adjustment policy


In [ ]:
if POLICY_SOURCE_PACKAGE_VERSION is None:
    policy = ManualAdjustmentPolicy.from_rows(
        name=POLICY_NAME,
        version=POLICY_VERSION,
        reason=POLICY_REASON,
        rows=MANUAL_ADJUSTMENTS,
        carry_forward=CARRY_FORWARD,
    )
else:
    if str(reviewed.technical.get("model_kind") or "").upper() == "MANUAL_EDIT":
        raise ValueError(
            "Replay a carry-forward policy onto a clean non-MANUAL_EDIT "
            "candidate so its multiplier is applied exactly once."
        )
    policy_source = load_model_version(
        pricing,
        model=model,
        package_version=int(POLICY_SOURCE_PACKAGE_VERSION),
    )
    policy = manual_adjustment_policy_from_candidate(
        policy_source,
        require_carry_forward=True,
    )
display(policy.table())
display(
    {
        "Policy": policy.name,
        "Policy version": policy.version,
        "Carry forward": policy.carry_forward,
        "Policy SHA-256": policy.sha256,
    }
)

## Apply and review before anything is published


In [ ]:
manual_review = apply_manual_adjustment_policy(reviewed, policy)
display(manual_review.rules)
display(manual_review.impact)
manual_review.edited_model

## Publish the immutable MANUAL_EDIT child


In [ ]:
manual_published = publish_manual_adjustment(
    pricing,
    review=manual_review,
)
display(
    {
        "Kind": manual_published.model_kind,
        "Package": manual_published.package_version,
        "Parent package ID": manual_published.parent_rate_package_id,
        "State": manual_published.package_status,
        "Reused equivalent": manual_published.deduplicated,
    }
)

## Optional explicit deployment


In [ ]:
if DEPLOY_AFTER_PUBLISH:
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the live package.")
    deployment_candidate = load_model_version(
        pricing,
        model=model,
        package_version=manual_published.package_version,
    )
    deployment = deploy_model_version(
        pricing,
        package=deployment_candidate,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)
else:
    display("Published only. Use notebook 06 when it is approved for deployment.")